# Prepare CDRH3 variant data from Hugging Face

Download the YM_0989 dataset, select HER2 measurements, filter the records, and extract the CDRH3 variant region.

In [1]:
import pandas as pd
from datasets import load_dataset

## Load the source data

In [ ]:
# ----------------------------
# Load the YM_0989 dataset from Hugging Face.
# ----------------------------
ds = load_dataset("aalphabio/open-alphaseq", "YM_0989", split="train")
df = ds.to_pandas()

In [3]:
print("raw shape:", df.shape)
print(df.columns.tolist())
df

raw shape: (94604, 10)
['mata_description', 'matalpha_description', 'mata_sequence', 'matalpha_sequence', 'alphaseq_affinity', 'affinity_lower_bound', 'affinity_upper_bound', 'normalized_affinity', 'above_background', 'sufficient_replicate_observations']


,mata_description,matalpha_description,mata_sequence,matalpha_sequence,alphaseq_affinity,affinity_lower_bound,affinity_upper_bound,normalized_affinity,above_background,sufficient_replicate_observations
0,ANeg1,AlphaNeg1,None,None,4.495806,4.664215,4.327396,-1.412818,False,True
1,ANeg1,AlphaNeg2,None,None,4.200819,4.417133,3.984505,-0.805647,False,True
2,ANeg1,AlphaNeg3,None,None,4.818761,5.054041,4.583482,0.870496,False,True
3,ANeg1,HER2-OPT_AGA2-POI,None,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,4.739934,4.967047,4.512822,0.799617,False,True
4,ANeg2,AlphaNeg1,None,None,4.698541,4.899842,4.497240,-0.182392,False,True
...,...,...,...,...,...,...,...,...,...,...
94599,TrastuzumabCDR_label_encoded_warm_99_d2,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.861273,1.910060,1.812487,-1.012774,True,True
94600,TrastuzumabCDR_label_encoded_warm_9_d2,AlphaNeg1,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,None,4.738377,4.993041,4.483714,1.309948,False,True
94601,TrastuzumabCDR_label_encoded_warm_9_d2,AlphaNeg2,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,None,4.280616,4.567948,3.993284,0.259591,False,True
94602,TrastuzumabCDR_label_encoded_warm_9_d2,AlphaNeg3,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,None,4.176565,4.353474,3.999657,-2.330465,False,True


## Select HER2 measurements

In [4]:
# ----------------------------
# Select HER2 measurements.
# ----------------------------
# The HER2 target is annotated as HER2-OPT_AGA2-POI.
her2_df = df[df["matalpha_description"] == "HER2-OPT_AGA2-POI"].copy()

print("HER2-only shape:", her2_df.shape)

HER2-only shape: (23651, 10)


In [5]:
# Count rows that satisfy both measurement quality criteria.
print("above_background & sufficient_replicate_observations:", her2_df[(her2_df["above_background"] == True) & (her2_df["sufficient_replicate_observations"] == True)].shape[0])

above_background & sufficient_replicate_observations: 12024


## Apply measurement quality filters

In [6]:
# ----------------------------
# Select measurements satisfying both quality criteria.
# ----------------------------
# Retain rows marked both above background and with sufficient replicate observations.
filtered_df = her2_df[
    (her2_df["above_background"] == True) &
    (her2_df["sufficient_replicate_observations"] == True)
].copy()

print("HER2 + high-confidence shape:", filtered_df.shape)

HER2 + high-confidence shape: (12024, 10)


In [7]:
# Count missing affinity values.
print(filtered_df["alphaseq_affinity"].isnull().sum())
# Count missing sequence values.
print(filtered_df["mata_sequence"].isnull().sum())
# Count missing affinity bounds.
print(filtered_df["affinity_upper_bound"].isnull().sum())
print(filtered_df["affinity_lower_bound"].isnull().sum())

0
0
0
0


## Prepare sequence and affinity records

In [8]:
# ----------------------------
# Prepare the data for regression.
# ----------------------------
# Use alphaseq_affinity as the target score.
# Exclude missing affinity values.
filtered_df = filtered_df[filtered_df["alphaseq_affinity"].notna()].copy()

# Exclude missing sequences.
filtered_df = filtered_df[filtered_df["mata_sequence"].notna()].copy()

# Exclude rows with missing upper or lower affinity bounds.
filtered_df = filtered_df[
    (filtered_df["affinity_upper_bound"].notna()) &
    (filtered_df["affinity_lower_bound"].notna())
].copy()

In [9]:
filtered_df 

,mata_description,matalpha_description,mata_sequence,matalpha_sequence,alphaseq_affinity,affinity_lower_bound,affinity_upper_bound,normalized_affinity,above_background,sufficient_replicate_observations
15,TrastuzumabCDR_WT_0,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.816049,1.841912,1.790185,-1.614151,True,True
19,TrastuzumabCDR_WT_1,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.901537,1.925614,1.877461,-0.866855,True,True
23,TrastuzumabCDR_WT_10,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.870567,1.897184,1.843951,-1.538581,True,True
27,TrastuzumabCDR_WT_11,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.924111,1.962477,1.885746,-0.638874,True,True
31,TrastuzumabCDR_WT_12,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.969507,2.014520,1.924494,-0.755557,True,True
...,...,...,...,...,...,...,...,...,...,...
94579,TrastuzumabCDR_label_encoded_warm_995_d3,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.998163,2.038720,1.957607,-0.783109,True,True
94583,TrastuzumabCDR_label_encoded_warm_996_d3,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,3.004830,3.108274,2.901386,-0.280042,True,True
94587,TrastuzumabCDR_label_encoded_warm_997_d3,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,2.583998,2.631388,2.536608,-1.154922,True,True
94599,TrastuzumabCDR_label_encoded_warm_99_d2,HER2-OPT_AGA2-POI,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,ACHQLCARGHCSGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYV...,1.861273,1.910060,1.812487,-1.012774,True,True


In [10]:
original = filtered_df.copy()

first_df = (
    original
    .drop_duplicates(subset="mata_sequence", keep="first")
    [["mata_sequence", "alphaseq_affinity"]]
    .rename(columns={"alphaseq_affinity": "affinity_first"})
)

agg_df = (
    original.groupby("mata_sequence", as_index=False)
    .agg(
        affinity_mean=("alphaseq_affinity", "mean"),
        affinity_median=("alphaseq_affinity", "median"),
        affinity_std=("alphaseq_affinity", "std"),
        n_measurements=("alphaseq_affinity", "size"),
    )
)

comparison = first_df.merge(agg_df, on="mata_sequence", how="left")

comparison["first_minus_mean"] = (
    comparison["affinity_first"] - comparison["affinity_mean"]
)
comparison["first_minus_median"] = (
    comparison["affinity_first"] - comparison["affinity_median"]
)

duplicated_only = comparison[comparison["n_measurements"] > 1]

print(duplicated_only[
    [
        "n_measurements",
        "first_minus_mean",
        "first_minus_median",
        "affinity_std",
    ]
].describe())

       n_measurements  first_minus_mean  first_minus_median  affinity_std
count     1863.000000       1863.000000         1863.000000   1863.000000
mean         2.305958          0.000471            0.002389      0.126011
std          1.246177          0.122340            0.125899      0.108168
min          2.000000         -0.591373           -0.740818      0.000037
25%          2.000000         -0.065933           -0.060790      0.049956
50%          2.000000         -0.000942            0.000000      0.100447
75%          2.000000          0.069499            0.066337      0.170416
max         50.000000          0.737759            1.025221      0.962409


In [11]:
# Select the sequence and affinity columns.
cdr3_data_clean = filtered_df[["mata_sequence", "alphaseq_affinity"]].copy()

# Rename the columns to seq and score.
cdr3_data_clean.rename(columns={"mata_sequence": "seq", "alphaseq_affinity": "score"}, inplace=True)

# Count duplicate sequences.
duplicate_count = cdr3_data_clean["seq"].duplicated().sum()
print(f"Number of duplicate 'seq' entries: {duplicate_count}")

# Remove duplicate sequences.
cdr3_data_clean.drop_duplicates(subset="seq", inplace=True)
cdr3_data_clean.reset_index(drop=True, inplace=True)

Number of duplicate 'seq' entries: 2433


In [12]:
cdr3_data_clean

,seq,score
0,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.816049
1,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.256693
2,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.741391
3,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.598910
4,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.292460
...,...,...
9586,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.998163
9587,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,3.004830
9588,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.583998
9589,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.861273


In [13]:
# Inspect the minimum and maximum sequence lengths.
min_length = cdr3_data_clean['seq'].apply(len).min()
max_length = cdr3_data_clean['seq'].apply(len).max()
print(f"Minimum seq length: {min_length}")
print(f"Maximum seq length: {max_length}")

Minimum seq length: 242
Maximum seq length: 246


In [14]:
cdr3_data_clean

,seq,score
0,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.816049
1,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.256693
2,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.741391
3,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.598910
4,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.292460
...,...,...
9586,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.998163
9587,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,3.004830
9588,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.583998
9589,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.861273


## Extract the CDRH3 variant region

### Wild-type variant region

`CSRWGGDGFYAMDYW`

In [15]:
cdr3_data_clean['seq'][0]

'EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIKR'

In [16]:
# Count sequences containing the wild-type prefix.
wt_prefix_seq = cdr3_data_clean['seq'].apply(lambda x: "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYY" in x).sum()
print(f"Number of rows containing the wt prefix seq: {wt_prefix_seq}")

Number of rows containing the wt prefix seq: 9591


In [17]:
# Count sequences containing the wild-type suffix.
wt_suffix_seq = cdr3_data_clean['seq'].apply(lambda x: "GQGTLVTVSSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIKR" in x).sum()
print(f"Number of rows containing the wt suffix seq: {wt_suffix_seq}")

Number of rows containing the wt suffix seq: 9591


In [18]:
# Extract the CDRH3 sequence between the wild-type prefix and suffix.
cdr3_data_clean['cdrh3_seq'] = cdr3_data_clean['seq'].apply(lambda x: x[x.find("EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYY") + len("EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYY"):x.find("GQGTLVTVSSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIKR")])

In [19]:
cdr3_data_clean['cdrh3_seq']

0       CSRWGGDGFYAMDYW
1       CSRWGGPGFYAFDYW
2       CSRWGGLGFYAFKYW
3       CSRWLGLGFYAFDYW
4       CSRWEGPGFYAFDYW
             ...       
9586    CSRWRSDFFYAMDYW
9587    CSRWEILGFYAMDYW
9588    CSRWELHGFYAMDYW
9589    CSRWSADGFYAMDYW
9590    CSRWGGDGPYAHDYW
Name: cdrh3_seq, Length: 9591, dtype: object

In [20]:
# Inspect the minimum and maximum CDRH3 sequence lengths.
cdrh3_min_length = cdr3_data_clean['cdrh3_seq'].apply(len).min()
cdrh3_max_length = cdr3_data_clean['cdrh3_seq'].apply(len).max()
print(f"Minimum cdrh3_seq length: {cdrh3_min_length}")
print(f"Maximum cdrh3_seq length: {cdrh3_max_length}")

Minimum cdrh3_seq length: 14
Maximum cdrh3_seq length: 18


In [21]:
cdr3_data_clean

,seq,score,cdrh3_seq
0,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.816049,CSRWGGDGFYAMDYW
1,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.256693,CSRWGGPGFYAFDYW
2,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.741391,CSRWGGLGFYAFKYW
3,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.598910,CSRWLGLGFYAFDYW
4,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.292460,CSRWEGPGFYAFDYW
...,...,...,...
9586,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.998163,CSRWRSDFFYAMDYW
9587,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,3.004830,CSRWEILGFYAMDYW
9588,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,2.583998,CSRWELHGFYAMDYW
9589,EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLE...,1.861273,CSRWSADGFYAMDYW


## Save the prepared data

In [ ]:
# Save the data.
plm_data = "...\\pLM-bVAE\\tutorial_pLM-bVAE_QA_Protein_Design\\data\\"
cdr3_data_clean.to_csv(plm_data + 'raw_data\\cdrh3_data_clean.csv', index=False)